# Zero-Dollar Speedrun: re-score the saved final weights

Pushed by `lanes/kaggle_push.py rescore`. Loads the final checkpoint from the attached training session's output
and scores it on the upstream val set in a fresh process with `scripts/rescore.py` (the trainer's own model and
`validate`): once in fp16 autocast as in training, once in fp32. Also hashes the checkpoints that join the two
sessions. Writes `/kaggle/working/rescore.json`.

In [ ]:
import glob, hashlib, json, os, shutil, signal, subprocess, sys, tarfile, time
T0 = time.time()
CFG = json.loads('{"ckpt": "zds-full-s2/run/ckpt/ckpt_0001451.pt", "expect": 3.2698315009167063, "hash_files": ["zds-full-s1/run/ckpt/ckpt_0001110.pt", "zds-full-s2/run/ckpt/ckpt_0001435.pt"], "kernel": "bharatbheesetti/zds-rescore-s2", "mode": "rescore", "model_file": "zds_final_model_step1450.pt", "scratch": "/kaggle/tmp", "session_cap_h": 2.0, "sources": ["bharatbheesetti/zds-full-s2", "bharatbheesetti/zds-full-s1"]}')
print(json.dumps(CFG, indent=1))
WORK = "/kaggle/working"
INPUT = CFG.get("input_root", "/kaggle/input")
SCRATCH = CFG.get("scratch", "/kaggle/tmp")
SRC = os.path.join(SCRATCH, "zds_src")
DATA = os.path.join(SCRATCH, "fineweb10B")
os.makedirs(DATA, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "x", N_GPU)
assert torch.cuda.is_available(), "GPU not attached"
os.chdir(WORK)

In [ ]:
# Kaggle auto-extracts uploaded archives: the dataset shows up either as zds_code.tar.gz or as a zds_code/ dir.
tarballs = sorted(glob.glob(os.path.join(INPUT, "**", "zds_code.tar.gz"), recursive=True))
extracted = sorted(glob.glob(os.path.join(INPUT, "**", "zds_code", "trainer", "train_t4.py"), recursive=True))
assert tarballs or extracted, ("zds code not found under /kaggle/input (neither zds_code.tar.gz nor zds_code/): "
                               "is the dataset attached (dataset_sources)? " + repr(glob.glob(os.path.join(INPUT, "**"), recursive=True)[:40]))
shutil.rmtree(SRC, ignore_errors=True)
if tarballs:
    os.makedirs(SRC)
    with tarfile.open(tarballs[0]) as tf:
        try:
            tf.extractall(SRC, filter="data")
        except TypeError:                  # python < 3.11.4 has no filter=
            tf.extractall(SRC)
    code_root = os.path.dirname(tarballs[0])
else:
    code_dir = os.path.dirname(os.path.dirname(extracted[0]))
    shutil.copytree(code_dir, SRC)
    code_root = os.path.dirname(code_dir)
manifest = glob.glob(os.path.join(code_root, "manifest.json"))
if manifest:
    print("code git sha:", json.load(open(manifest[0])).get("git_sha"))
print(sorted(os.listdir(SRC)), os.listdir(os.path.join(SRC, "trainer")))

In [ ]:
t = time.time()
subprocess.run([sys.executable, "-u", "scripts/get_data.py", "0", "--dir", DATA], cwd=SRC, check=True)   # val shard only
print(f"val shard ready in {time.time() - t:.0f} s:", sorted(os.listdir(DATA)))

In [ ]:
def find(rel):
    """An attached kernel output file by its tail path, e.g. zds-full-s2/run/ckpt/ckpt_0001451.pt."""
    hits = [p for p in glob.glob(os.path.join(INPUT, "**", os.path.basename(rel)), recursive=True) if p.endswith("/" + rel)]
    return sorted(hits)[0] if hits else None

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 24):
            h.update(chunk)
    return h.hexdigest()

CKPT = find(CFG["ckpt"])
assert CKPT, f"{CFG['ckpt']} not found under {INPUT}: " + repr(glob.glob(os.path.join(INPUT, "**"), recursive=True)[:60])
side = json.load(open(CKPT[:-3] + ".json"))
assert os.path.getsize(CKPT) == side["bytes"], "checkpoint size does not match its sidecar (not committed)"
print("final checkpoint:", CKPT, side["bytes"], "bytes, step", side["step"], side["reason"])
HASHES = {}
for rel in CFG.get("hash_files", []):
    p = find(rel)
    if p is None:
        HASHES[rel] = None
        continue
    t = time.time()
    js = p[:-3] + ".json"
    HASHES[rel] = {"bytes": os.path.getsize(p), "sha256": sha256(p),
                   "sidecar": json.load(open(js)) if os.path.exists(js) else None}
    print(rel, HASHES[rel]["sha256"], f"{time.time() - t:.0f} s")

In [ ]:
CAP_S = CFG["session_cap_h"] * 3600

def rescore(amp, extra, margin_min):
    out_json = os.path.join(WORK, f"rescore_{amp}.json")
    out_log = os.path.join(WORK, f"rescore_{amp}.out")
    cmd = ["torchrun", "--standalone", f"--nproc_per_node={max(N_GPU, 1)}", "scripts/rescore.py", "--ckpt", CKPT,
           "--data-dir", DATA, "--device", CFG.get("device", "cuda"), "--amp", amp, "--expect", repr(CFG["expect"]),
           "--out", out_json] + extra + [str(x) for x in CFG.get("rescore_args", [])]
    print(" ".join(cmd), flush=True)
    timeout = CAP_S - (time.time() - T0) - margin_min * 60
    with open(out_log, "w") as log:
        proc = subprocess.Popen(cmd, cwd=SRC, stdout=log, stderr=subprocess.STDOUT, text=True, start_new_session=True,
                                env=dict(os.environ, PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4"))
        try:
            rc = proc.wait(timeout=max(timeout, 60))
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            rc = "timeout"
    print(f"[{amp}] exit {rc}; tail:\n" + "\n".join(open(out_log).read().splitlines()[-12:]), flush=True)
    return rc, (json.load(open(out_json)) if os.path.exists(out_json) else None)

RC16, R16 = rescore("fp16", ["--hash", "--save-model", os.path.join(WORK, CFG["model_file"])], margin_min=25)
RC32, R32 = rescore("fp32", [], margin_min=8)

In [ ]:
man = glob.glob(os.path.join(INPUT, "**", "manifest.json"), recursive=True)
summary = {
    "kernel": CFG["kernel"], "sources": CFG["sources"], "ckpt": CFG["ckpt"], "expect": CFG["expect"],
    "code_git_sha": json.load(open(man[0])).get("git_sha") if man else None,
    "fp16": {"exit": RC16, "result": R16}, "fp32": {"exit": RC32, "result": R32}, "hashes": HASHES,
    "ckpt_sidecar": side, "n_gpu": N_GPU, "gpu": torch.cuda.get_device_name(0) if N_GPU else "cpu", "torch": torch.__version__,
    "elapsed_h": round((time.time() - T0) / 3600, 3),
}
json.dump(summary, open(os.path.join(WORK, "rescore.json"), "w"), indent=1)
shutil.rmtree(SRC, ignore_errors=True)
print(json.dumps({k: v for k, v in summary.items() if k not in ("ckpt_sidecar",)}, indent=1)[:6000])
assert RC16 == 0, f"fp16 re-score failed: {RC16}"
